# 03 · 리포트 (SR ±CI, pooled seed + jerk + horizon)
학습중 eval(eval_step json) 기반 **SR 곡선 + seed별 best + pooled Wilson CI** + best 체크포인트 **jerk** 표/그림.
출력 → `outputs/v23/final_report/`. 실로봇(pick&place/sorting/stacking) 수치는 은지님 결과로 표 채우기.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import common_final as cf
print('FINAL_TAGS:', cf.FINAL_TAGS, '| seeds:', cf.FINAL_SEEDS)
print('OUTPUT_BASE:', cf.OUTPUT_BASE)
import re, json, glob, csv
import numpy as np
import matplotlib; matplotlib.use('Agg') if not os.environ.get('DISPLAY') else None
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,'savefig.bbox':'tight','font.size':13,
  'axes.titleweight':'bold','axes.grid':True,'grid.alpha':0.3,'axes.spines.top':False,'axes.spines.right':False})
_kf = next((c for c in ['NanumGothic','Malgun Gothic','AppleGothic','UnDotum'] if c in {f.name for f in fm.fontManager.ttflist}), None)
if _kf: plt.rcParams['font.family'] = _kf
plt.rcParams['axes.unicode_minus'] = False
TASK = cf.PRIMARY_SIM      # 'libero_10' (메인 sim). transfer 보려면 'transfer'로 바꿔 재실행.
REP = cf.OUTPUT_BASE / f'final_report_{TASK}'; REP.mkdir(parents=True, exist_ok=True)
print('report out:', REP, '| KR font:', _kf, '| fps:', cf.fps_of(TASK))

In [ ]:
# 학습중 eval json 읽기: {step: {sr, k, n}} (successes로 Wilson CI)
def read_steps(tag, seed):
    d = cf.train_dir(tag, seed, TASK) / 'eval'
    out = {}
    for p in glob.glob(str(d / 'eval_step_*.json')):
        m = re.search(r'eval_step_(\d+)\.json', p)
        if not m: continue
        try: j = json.loads(Path(p).read_text())
        except Exception: continue
        sr = j.get('overall', {}).get('pc_success')
        if sr is None: continue
        succ = []
        for pt in j.get('per_task', []):
            s = pt.get('metrics', {}).get('successes')
            if s: succ.extend(s)
        n = len(succ) if succ else int(j['overall'].get('n_episodes') or 0)
        k = int(sum(bool(x) for x in succ)) if succ else int(round(sr/100*n))
        out[int(m.group(1))] = {'sr': float(sr), 'k': k, 'n': n}
    return dict(sorted(out.items()))

# seed별 best SR 표 + seed 평균 + pooled Wilson CI
fmt = lambda x, s='%.1f': (s % x) if isinstance(x,(int,float)) and np.isfinite(x) else '  -'
rows = []
for t in cf.FINAL_TAGS:
    per_seed, pk, pn = {}, 0, 0
    for s in cf.FINAL_SEEDS:
        c = read_steps(t, s)
        if not c: continue
        bs = max(c, key=lambda x: (c[x]['sr'], x)); bd = c[bs]
        per_seed[s] = bd['sr']; pk += bd['k']; pn += bd['n']
    srs = list(per_seed.values())
    lo, hi = (cf.wilson_ci(pk, pn) if pn else (None, None))
    rows.append(dict(tag=t, label=cf.FINAL_LABELS[t], nseed=len(srs), per=per_seed,
        mean=(np.mean(srs) if srs else None), std=(np.std(srs, ddof=1) if len(srs)>1 else 0.0),
        plo=(lo*100 if lo is not None else None), phi=(hi*100 if hi is not None else None)))
print('%-30s %6s %6s %6s %16s' % ('MODEL','mean','std','#seed','pooled95%CI'))
print('-'*72)
for r in rows:
    ci = '[%s,%s]' % (fmt(r['plo']), fmt(r['phi'])) if r['plo'] is not None else '  -'
    print('%-30s %6s %6s %6d %16s' % (r['label'][:30], fmt(r['mean']), fmt(r['std']), r['nseed'], ci))
with open(REP/'sr_pooled.csv','w',newline='') as fp:
    w = csv.writer(fp); w.writerow(['tag','label','mean_sr','std','n_seed','pooled_ci_lo','pooled_ci_hi','per_seed'])
    for r in rows: w.writerow([r['tag'],r['label'],r['mean'],r['std'],r['nseed'],r['plo'],r['phi'],r['per']])
print('saved:', REP/'sr_pooled.csv')

In [ ]:
# SR vs step 곡선 (seed 평균)
fig, ax = plt.subplots(figsize=(10,6))
for t in cf.FINAL_TAGS:
    curves = [read_steps(t, s) for s in cf.FINAL_SEEDS]
    curves = [c for c in curves if c]
    if not curves: continue
    steps = sorted(set().union(*[set(c) for c in curves]))
    mean = [np.mean([c[st]['sr'] for c in curves if st in c]) for st in steps]
    ax.plot([s/1000 for s in steps], mean, '-o', ms=5, color=cf.COLOR.get(t,'#333'),
            label=f'{cf.FINAL_LABELS[t]} (best {max(mean):.0f}%)')
ax.set_xlabel('step (k)'); ax.set_ylabel('Success Rate (%)')
ax.set_title('학습중 eval SR (seed 평균) — 최종 4모델' if _kf else 'In-training SR (seed mean)')
ax.legend(frameon=False, fontsize=10)
for e in ('png','pdf'): fig.savefig(REP/f'sr_curves.{e}')
print('saved sr_curves'); plt.show()

In [ ]:
# best 체크포인트 jerk 표 (학습중 eval action_logs)
import smooth_metrics as sm
def best_step(tag, seed):
    c = read_steps(tag, seed)
    return max(c, key=lambda x: (c[x]['sr'], x)) if c else None
def load_pt(adir, cap=100):
    import torch; eps=[]
    if adir is None or not Path(adir).is_dir(): return eps
    for f in sorted(Path(adir).glob('episode_*.pt'))[:cap]:
        try: o = torch.load(f, map_location='cpu', weights_only=False)
        except Exception: continue
        a = o['actions'] if isinstance(o, dict) and 'actions' in o else o
        if hasattr(a,'detach'): a = a.detach().cpu().numpy()
        a = np.asarray(a, float)
        if a.ndim==2: eps.append(a)
        elif a.ndim==3: eps += [a[i] for i in range(a.shape[0])]
    return eps
jr = []
for t in cf.FINAL_TAGS:
    s0 = cf.FINAL_SEEDS[0]; bs = best_step(t, s0)
    adir = cf.train_dir(t, s0, TASK)/'eval'/f'videos_step_{bs}'/'action_logs' if bs else None
    trajs = load_pt(adir)
    if not trajs: jr.append({'tag':t}); continue
    K = cf.MODEL_CONFIGS[t][2]; agg = sm.aggregate_smoothness(trajs, K, fs=cf.fps_of(TASK))
    jr.append({'tag':t,'contrast':agg.get('boundary_jerk_contrast_mean'),'ratio':agg.get('boundary_jerk_ratio_mean'),
               'ij':agg.get('interior_jerk_mean'),'sparc':agg.get('sparc_mean')})
print('%-30s %10s %8s %10s %8s' % ('MODEL','contrast','ratio','i_jerk','SPARC'))
print('-'*70)
for r in jr:
    if 'contrast' not in r: print('%-30s  (action_logs 없음)'%cf.FINAL_LABELS[r['tag']][:30]); continue
    print('%-30s %10s %8s %10s %8s'%(cf.FINAL_LABELS[r['tag']][:30],fmt(r['contrast'],'%.4f'),fmt(r['ratio'],'%.2f'),fmt(r['ij'],'%.4f'),fmt(r['sparc'],'%.2f')))
with open(REP/'jerk.csv','w',newline='') as fp:
    w=csv.writer(fp); w.writerow(['tag','contrast','ratio','interior_jerk','sparc'])
    for r in jr: w.writerow([r['tag'],r.get('contrast'),r.get('ratio'),r.get('ij'),r.get('sparc')])
print('saved:', REP/'jerk.csv')

## Horizon 그림 (킬러 그림) — 표 채우기
실로봇 SR을 은지님 결과로 채우면 **task 길수록 우리-ACT 격차↑** 그림 완성.
아래 `REAL` 딕셔너리에 은지님 SR(%) 넣으면 그림 자동 생성.

In [ ]:
# task horizon 축: 짧은->긴. 값 채우면 그림 생성 (없으면 sim만).
# 형식: REAL[task] = {'act': SR, 'ours': SR}  (ours=최종후보)
REAL = {
  'pick&place': {'act': None, 'ours': None},   # 짧음(동급)
  'sorting':    {'act': None, 'ours': None},   # 김(우세)
  'stacking':   {'act': None, 'ours': None},   # 정밀(우세)
}
order = [k for k in REAL if REAL[k]['act'] is not None and REAL[k]['ours'] is not None]
if order:
    fig, ax = plt.subplots(figsize=(8,5))
    x = range(len(order))
    ax.plot(list(x), [REAL[k]['act'] for k in order], '-o', color='#000', label='ACT')
    ax.plot(list(x), [REAL[k]['ours'] for k in order], '-o', color='#2ca02c', label='Ours (최종)')
    ax.set_xticks(list(x)); ax.set_xticklabels(order)
    ax.set_ylabel('Success Rate (%)'); ax.set_xlabel('task (짧은→긴 horizon)')
    ax.set_title('SR vs task horizon (길수록 격차↑)' if _kf else 'SR vs horizon')
    ax.legend(frameon=False)
    for e in ('png','pdf'): fig.savefig(REP/f'horizon.{e}')
    print('saved horizon'); plt.show()
else:
    print('REAL 값 비어있음 — 은지님 실로봇 SR 채우면 그림 생성됨. 지금은 sim SR 표만 사용.')